# NA2M — concurvity filter: hyperparameter tuning + single run

One fixed train/val/test split. Two-stage tuning (Stage 1 mains → Stage 2 marginal clarity),
then a single arm-C run (interactions + concurvity filter) on that same split.

In [ ]:
import random
import numpy as np
import torch

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

SEED = 42
set_seed(SEED)

## Data

In [ ]:
from na2m.data.data_utils import load_compas, preprocess

df = load_compas(r'C:\Users\maart\OneDrive\Documenten\Universiteit\Scriptie\python_repo\thesis-nam\datasets\raw\compas-scores-two-years.csv')
X, y, feature_meta = preprocess(df)

print(f'Samples: {X.shape[0]}, Features: {X.shape[1]}')
for fm in feature_meta:
    print(f'  {fm.name:25s} type={fm.type}')

## Predetermined split

One fixed, seeded train/val/test split. Both tuning stages and the final run use this exact split.

In [ ]:
from pathlib import Path
from na2m.data.data_utils import split
from na2m.utils.config import load_na2m_search_config

SEARCH_CONFIG = r'C:\Users\maart\OneDrive\Documenten\Universiteit\Scriptie\python_repo\thesis-nam\configs\compas_na2m_search.yaml'
REPO = Path(SEARCH_CONFIG).parent.parent

fixed_params, search_space = load_na2m_search_config(SEARCH_CONFIG)

X_train, X_val, X_test, y_train, y_val, y_test = split(
    X, y, fixed_params['val_frac'], fixed_params['test_frac'], seed=SEED
)
print(f'Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}')

# Single tuned-config artifact: Stage 1 writes it, Stage 2 augments it with clarity.
TUNED_PATH = Path(SEARCH_CONFIG).parent / 'compas-scores-two-years_na2m_tuned.yaml'

### Load the tuning scripts

`scripts/` is not an importable package here, so load the two tuning modules directly by file path.

In [ ]:
import importlib.util

def _load_script(name, filename):
    spec = importlib.util.spec_from_file_location(name, REPO / 'scripts' / 'na2m' / filename)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

tune_na2m    = _load_script('tune_na2m_script', 'tune_na2m.py')
tune_clarity = _load_script('tune_clarity_script', 'tune_clarity.py')

## Stage 1 — tune main-effect hyperparameters

Searches mains hp (lr, dropout, regularization, activation, …) with interactions off;
MedianPruner cuts weak trials early. Writes the tuned config to `TUNED_PATH` (clarity stays 0.0).

In [ ]:
tune_na2m.tune_fold(
    fixed_params, search_space, feature_meta,
    X_train, y_train, X_val, y_val,
    TUNED_PATH,
    study_name='concurvity_nb_main_search',
)

## Stage 2 — tune marginal-clarity (λ)

Reads the Stage-1 tuned config, trains the mains **once**, then searches `clarity_regularization`
over Stage 2/3 only (each trial restarts from the same fixed mains). Writes the best λ back into `TUNED_PATH`.

In [ ]:
clarity_n_trials, clarity_search_spec = tune_clarity.load_clarity_search_config(SEARCH_CONFIG)

tune_clarity.tune_clarity_fold(
    TUNED_PATH,
    clarity_n_trials,
    clarity_search_spec,
    feature_meta,
    X_train, y_train, X_val, y_val,
    study_name='concurvity_nb_clarity_search',
)

## Tuned config (mains hp + best λ)

In [ ]:
from na2m.utils.config import load_na2m_config

config = load_na2m_config(str(TUNED_PATH))
print(config)

## Loaders

In [ ]:
from na2m.data.dataset import NAMDataset
from torch.utils.data import DataLoader

train_loader = DataLoader(NAMDataset(X_train, y_train), batch_size=config.batch_size, shuffle=True)
val_loader   = DataLoader(NAMDataset(X_val, y_val),     batch_size=config.batch_size, shuffle=False)
pool_loader  = DataLoader(
    NAMDataset(np.concatenate([X_train, X_val]), np.concatenate([y_train, y_val])),
    batch_size=config.batch_size, shuffle=False,
)
test_loader  = DataLoader(NAMDataset(X_test, y_test),   batch_size=config.batch_size, shuffle=False)

print(f'Train: {len(X_train)}, Val: {len(X_val)}, Pool: {len(X_train) + len(X_val)}, Test: {len(X_test)}')

## Run arm C — interactions + concurvity filter

In [ ]:
from na2m.models.na2m import NA2M
from na2m.training.fit_na2m import fit_na2m

set_seed(SEED)
model_c = NA2M(
    num_features=X.shape[1], feature_meta=feature_meta,
    num_units=config.num_units, hidden_sizes=config.hidden_sizes,
    dropout=config.dropout, feature_dropout=config.feature_dropout,
    activation=config.activation, inter_units=config.inter_units,
    inter_hidden=config.inter_hidden,
)

result_c = fit_na2m(
    model_c, train_loader, val_loader, pool_loader, config,
    with_interactions=True,
    with_concurvity_filter=True,
)

## Result

In [ ]:
from na2m.training.metrics import auroc

model_c.eval()
logits_all, targets_all = [], []
with torch.no_grad():
    for X_batch, y_batch, _ in test_loader:
        logits, _ = model_c(X_batch)
        logits_all.append(logits)
        targets_all.append(y_batch)
auc_c = auroc(torch.cat(logits_all), torch.cat(targets_all))

pairs = result_c['active_pairs']
print(f'Arm C (concurvity filter)  AUROC={auc_c:.4f}  interactions={len(pairs)}')
print(f'clarity_regularization = {config.clarity_regularization:.6f}')
for j, k in pairs:
    print(f'  {feature_meta[j].name} x {feature_meta[k].name}')